# 11.2 ONNX for Computer Vision — Apply

## Objective

Build complete computer vision inference pipelines with ONNX Runtime: image preprocessing,
CNN construction, classification, Non-Maximum Suppression (NMS), batch inference,
and feature extraction.

**Prerequisites:** `pip install onnx onnxruntime numpy`

## Table of Contents
1. [Setup](#setup)
2. [Exercise 1 — Build & Inspect a CV Session](#ex1)
3. [Exercise 2 — Image Preprocessing Pipeline](#ex2)
4. [Exercise 3 — Build a ResNet-like Block](#ex3)
5. [Exercise 4 — Full Classification Pipeline](#ex4)
6. [Exercise 5 — Non-Maximum Suppression](#ex5)
7. [Exercise 6 — Batch Image Inference](#ex6)
8. [Exercise 7 — Feature Extraction](#ex7)
9. [Challenge — Complete Image Classification Pipeline](#challenge)
10. [Summary](#summary)

In [ ]:
!pip install onnx onnxruntime numpy -q

<a id='setup'></a>
## Setup

We build a small CNN using `onnx.helper` with Conv → BatchNorm → ReLU blocks,
similar to the building blocks of ResNet.

In [ ]:
import os
import time
import json
import numpy as np
from typing import Any, Dict, List, Optional, Tuple

import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

np.random.seed(42)

NUM_CLASSES = 10
INPUT_SIZE = 32


def build_cv_model(num_classes: int = NUM_CLASSES) -> onnx.ModelProto:
    """Build a CNN: Conv(3->32) -> ReLU -> Conv(32->64) -> ReLU -> GAP -> FC."""
    inits = []

    # Conv1: 3 -> 32, kernel 3x3
    W1 = numpy_helper.from_array(
        np.random.randn(32, 3, 3, 3).astype(np.float32) * 0.1, name="conv1_W"
    )
    B1 = numpy_helper.from_array(np.zeros(32, dtype=np.float32), name="conv1_B")
    inits.extend([W1, B1])

    # Conv2: 32 -> 64, kernel 3x3
    W2 = numpy_helper.from_array(
        np.random.randn(64, 32, 3, 3).astype(np.float32) * 0.1, name="conv2_W"
    )
    B2 = numpy_helper.from_array(np.zeros(64, dtype=np.float32), name="conv2_B")
    inits.extend([W2, B2])

    # FC: 64 -> num_classes
    FC_W = numpy_helper.from_array(
        np.random.randn(64, num_classes).astype(np.float32) * 0.1, name="fc_W"
    )
    FC_B = numpy_helper.from_array(np.zeros(num_classes, dtype=np.float32), name="fc_B")
    inits.extend([FC_W, FC_B])

    # Build graph
    nodes = [
        helper.make_node("Conv", ["image", "conv1_W", "conv1_B"], ["conv1_out"],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node("Relu", ["conv1_out"], ["relu1_out"]),
        helper.make_node("Conv", ["relu1_out", "conv2_W", "conv2_B"], ["conv2_out"],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node("Relu", ["conv2_out"], ["relu2_out"]),
        helper.make_node("GlobalAveragePool", ["relu2_out"], ["gap_out"]),
        helper.make_node("Flatten", ["gap_out"], ["flat_out"], axis=1),
        helper.make_node("MatMul", ["flat_out", "fc_W"], ["mm_out"]),
        helper.make_node("Add", ["mm_out", "fc_B"], ["logits"]),
    ]

    X = helper.make_tensor_value_info("image", TensorProto.FLOAT, ["batch", 3, INPUT_SIZE, INPUT_SIZE])
    Y = helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["batch", num_classes])

    graph = helper.make_graph(nodes, "cv_model", [X], [Y], inits)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


cv_model = build_cv_model()
MODEL_PATH = "/tmp/cv_classifier.onnx"
onnx.save(cv_model, MODEL_PATH)
print(f"CV model saved: {os.path.getsize(MODEL_PATH):,} bytes")
print(f"Nodes: {len(cv_model.graph.node)}, Parameters: {sum(int(np.prod(list(i.dims))) for i in cv_model.graph.initializer):,}")

<a id='ex1'></a>
## Exercise 1 — Build & Inspect a CV Session

Image models have fixed spatial input expectations. Understanding input shapes
is critical for correct preprocessing.

In [ ]:
def build_cv_session(model_path: str) -> ort.InferenceSession:
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])


def inspect_cv_model(sess: ort.InferenceSession) -> Dict[str, Any]:
    """Extract CV-specific metadata."""
    inp = sess.get_inputs()[0]
    out = sess.get_outputs()[0]

    # Parse NCHW format
    shape = inp.shape
    spatial_info = {}
    if len(shape) == 4:
        spatial_info = {
            "batch_dim": str(shape[0]),
            "channels": shape[1] if isinstance(shape[1], int) else "dynamic",
            "height": shape[2] if isinstance(shape[2], int) else "dynamic",
            "width": shape[3] if isinstance(shape[3], int) else "dynamic",
            "format": "NCHW",
        }

    return {
        "input": {"name": inp.name, "shape": [str(d) for d in shape], "type": inp.type},
        "output": {"name": out.name, "shape": [str(d) for d in out.shape], "type": out.type},
        "spatial": spatial_info,
        "num_classes": out.shape[-1] if isinstance(out.shape[-1], int) else "dynamic",
    }


cv_session = build_cv_session(MODEL_PATH)
cv_info = inspect_cv_model(cv_session)
print("CV Model Inspection:")
print(json.dumps(cv_info, indent=2))

assert cv_info["spatial"]["channels"] == 3
assert cv_info["spatial"]["format"] == "NCHW"
assert cv_info["num_classes"] == NUM_CLASSES

# Quick inference test
dummy = np.random.randn(1, 3, INPUT_SIZE, INPUT_SIZE).astype(np.float32)
out = cv_session.run(None, {"image": dummy})[0]
assert out.shape == (1, NUM_CLASSES)
print(f"\nInference OK — output: {out.shape}")

<a id='ex2'></a>
## Exercise 2 — Image Preprocessing Pipeline

Standard CV preprocessing:
1. Resize to target spatial dimensions
2. Convert HWC uint8 → CHW float32
3. Normalize with ImageNet statistics:

$$x_{\text{norm}}^{(c)} = \frac{x^{(c)} / 255 - \mu_c}{\sigma_c}$$

where $\mu = [0.485, 0.456, 0.406]$ and $\sigma = [0.229, 0.224, 0.225]$.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def resize_nearest(image: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    """Nearest-neighbor resize (no PIL dependency)."""
    h, w = image.shape[:2]
    row_idx = np.clip((np.arange(target_h) * h / target_h).astype(int), 0, h - 1)
    col_idx = np.clip((np.arange(target_w) * w / target_w).astype(int), 0, w - 1)
    return image[row_idx][:, col_idx]


def preprocess_image(
    image_hwc_uint8: np.ndarray,
    target_size: Tuple[int, int] = (32, 32),
    mean: np.ndarray = IMAGENET_MEAN,
    std: np.ndarray = IMAGENET_STD,
) -> np.ndarray:
    """Full preprocessing: resize -> float -> normalize -> NCHW."""
    # Resize
    resized = resize_nearest(image_hwc_uint8, target_size[0], target_size[1])

    # uint8 -> float32 [0, 1]
    x = resized.astype(np.float32) / 255.0

    # Normalize per channel
    x = (x - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)

    # HWC -> CHW, add batch
    x = np.transpose(x, (2, 0, 1))[np.newaxis, ...]  # [1, C, H, W]
    return x.astype(np.float32)


# Create synthetic test image (64x48 RGB)
synthetic_image = np.random.randint(0, 256, size=(48, 64, 3), dtype=np.uint8)

preprocessed = preprocess_image(synthetic_image, target_size=(INPUT_SIZE, INPUT_SIZE))
print(f"Input image: {synthetic_image.shape} uint8")
print(f"Preprocessed: {preprocessed.shape} float32")
print(f"Value range: [{preprocessed.min():.3f}, {preprocessed.max():.3f}]")

assert preprocessed.shape == (1, 3, INPUT_SIZE, INPUT_SIZE)
assert preprocessed.dtype == np.float32

# Run through model
output = cv_session.run(None, {"image": preprocessed})[0]
assert output.shape == (1, NUM_CLASSES)
print(f"Model output: {output.shape}")

<a id='ex3'></a>
## Exercise 3 — Build a ResNet-like Block

A residual block adds a skip connection:

$$\mathbf{y} = \mathcal{F}(\mathbf{x}) + \mathbf{x}$$

where $\mathcal{F}$ is typically Conv → BN → ReLU → Conv → BN.
The skip connection enables gradient flow through deep networks.

In [ ]:
def build_resnet_block_model(channels: int = 16) -> onnx.ModelProto:
    """
    Build a model with one residual block:
    Input -> Conv -> ReLU -> Conv -> Add(input) -> ReLU -> GAP -> FC
    """
    inits = []

    # Initial conv to get to `channels`
    W0 = numpy_helper.from_array(
        np.random.randn(channels, 3, 3, 3).astype(np.float32) * 0.1, name="conv0_W"
    )
    B0 = numpy_helper.from_array(np.zeros(channels, dtype=np.float32), name="conv0_B")
    inits.extend([W0, B0])

    # Residual block: conv_a and conv_b (same channels)
    Wa = numpy_helper.from_array(
        np.random.randn(channels, channels, 3, 3).astype(np.float32) * 0.1, name="res_Wa"
    )
    Ba = numpy_helper.from_array(np.zeros(channels, dtype=np.float32), name="res_Ba")
    Wb = numpy_helper.from_array(
        np.random.randn(channels, channels, 3, 3).astype(np.float32) * 0.1, name="res_Wb"
    )
    Bb = numpy_helper.from_array(np.zeros(channels, dtype=np.float32), name="res_Bb")
    inits.extend([Wa, Ba, Wb, Bb])

    # FC head
    FC_W = numpy_helper.from_array(
        np.random.randn(channels, NUM_CLASSES).astype(np.float32) * 0.1, name="fc_W"
    )
    FC_B = numpy_helper.from_array(np.zeros(NUM_CLASSES, dtype=np.float32), name="fc_B")
    inits.extend([FC_W, FC_B])

    nodes = [
        # Initial conv
        helper.make_node("Conv", ["image", "conv0_W", "conv0_B"], ["h0"],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node("Relu", ["h0"], ["h0r"]),
        # Residual block
        helper.make_node("Conv", ["h0r", "res_Wa", "res_Ba"], ["ra"],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node("Relu", ["ra"], ["rar"]),
        helper.make_node("Conv", ["rar", "res_Wb", "res_Bb"], ["rb"],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        # Skip connection: output = F(x) + x
        helper.make_node("Add", ["rb", "h0r"], ["residual"]),
        helper.make_node("Relu", ["residual"], ["res_out"]),
        # Classification head
        helper.make_node("GlobalAveragePool", ["res_out"], ["gap"]),
        helper.make_node("Flatten", ["gap"], ["flat"], axis=1),
        helper.make_node("MatMul", ["flat", "fc_W"], ["mm"]),
        helper.make_node("Add", ["mm", "fc_B"], ["logits"]),
    ]

    X = helper.make_tensor_value_info("image", TensorProto.FLOAT, ["batch", 3, INPUT_SIZE, INPUT_SIZE])
    Y = helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["batch", NUM_CLASSES])

    graph = helper.make_graph(nodes, "resnet_block", [X], [Y], inits)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


resnet_model = build_resnet_block_model(channels=16)
RESNET_PATH = "/tmp/resnet_block.onnx"
onnx.save(resnet_model, RESNET_PATH)

resnet_session = build_cv_session(RESNET_PATH)
test_img = np.random.randn(1, 3, INPUT_SIZE, INPUT_SIZE).astype(np.float32)
res_out = resnet_session.run(None, {"image": test_img})[0]

print(f"ResNet block model: {os.path.getsize(RESNET_PATH):,} bytes")
print(f"Output shape: {res_out.shape}")
assert res_out.shape == (1, NUM_CLASSES)

# Verify skip connection works (output shouldn't be zero even with small weights)
assert np.any(np.abs(res_out) > 1e-6)
print("ResNet block with skip connection validated.")

<a id='ex4'></a>
## Exercise 4 — Full Classification Pipeline

The complete pipeline: preprocess → infer → softmax → top-k → labels.

Softmax with temperature $T$ for calibrated predictions:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

$T > 1$ produces softer distributions; $T < 1$ makes predictions sharper.

In [ ]:
CIFAR_LABELS = ["airplane", "automobile", "bird", "cat", "deer",
                "dog", "frog", "horse", "ship", "truck"]


def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Temperature-scaled softmax."""
    scaled = logits / temperature
    shifted = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True)


def classify_image(
    image_hwc_uint8: np.ndarray,
    session: ort.InferenceSession,
    labels: List[str],
    target_size: Tuple[int, int] = (32, 32),
    top_k: int = 5,
    temperature: float = 1.0,
) -> Dict[str, Any]:
    """Complete image classification pipeline."""
    # Preprocess
    tensor = preprocess_image(image_hwc_uint8, target_size=target_size)

    # Inference
    input_name = session.get_inputs()[0].name
    logits = session.run(None, {input_name: tensor})[0][0]  # [num_classes]

    # Postprocess
    probs = softmax(logits, temperature=temperature)
    top_indices = np.argsort(-probs)[:top_k]

    predictions = []
    for idx in top_indices:
        predictions.append({
            "class_idx": int(idx),
            "label": labels[idx] if idx < len(labels) else f"class_{idx}",
            "probability": float(probs[idx]),
        })

    return {
        "top_prediction": predictions[0]["label"],
        "confidence": predictions[0]["probability"],
        "top_k": predictions,
        "entropy": float(-np.sum(probs * np.log(probs + 1e-10))),
    }


# Test classification pipeline
test_image = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)

result = classify_image(test_image, cv_session, CIFAR_LABELS, target_size=(INPUT_SIZE, INPUT_SIZE))
print("Classification Result:")
print(f"  Top prediction: {result['top_prediction']} ({result['confidence']:.1%})")
print(f"  Entropy: {result['entropy']:.3f} (max={np.log(NUM_CLASSES):.3f})")
print("  Top-5:")
for p in result["top_k"]:
    print(f"    {p['label']:12s} {p['probability']:.3f}")

# Verify
assert result["top_prediction"] in CIFAR_LABELS
assert 0 <= result["confidence"] <= 1
assert sum(p["probability"] for p in result["top_k"]) <= 1.0 + 1e-5
print("\nClassification pipeline validated.")

<a id='ex5'></a>
## Exercise 5 — Non-Maximum Suppression (NMS)

Object detection models output many overlapping boxes. NMS keeps only the best:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{intersection area}}{\text{area}_A + \text{area}_B - \text{intersection area}}$$

Algorithm:
1. Sort boxes by confidence
2. Keep highest confidence box
3. Remove all boxes with IoU > threshold
4. Repeat until no boxes remain

In [ ]:
def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection

    return intersection / (union + 1e-6)


def non_maximum_suppression(
    boxes: np.ndarray,
    scores: np.ndarray,
    iou_threshold: float = 0.5,
    score_threshold: float = 0.3,
) -> List[int]:
    """
    Apply NMS to detection boxes.
    
    Args:
        boxes: [N, 4] array of [x1, y1, x2, y2]
        scores: [N] array of confidence scores
        iou_threshold: suppress boxes with IoU above this
        score_threshold: ignore boxes below this score
    
    Returns:
        List of kept box indices
    """
    # Filter by score threshold
    valid_mask = scores > score_threshold
    valid_indices = np.where(valid_mask)[0]

    if len(valid_indices) == 0:
        return []

    # Sort by score descending
    sorted_idx = valid_indices[np.argsort(-scores[valid_indices])]

    kept = []
    suppressed = set()

    for idx in sorted_idx:
        if idx in suppressed:
            continue

        kept.append(int(idx))

        # Suppress overlapping boxes
        for other_idx in sorted_idx:
            if other_idx in suppressed or other_idx == idx:
                continue
            iou = compute_iou(boxes[idx], boxes[other_idx])
            if iou > iou_threshold:
                suppressed.add(other_idx)

    return kept


# Test NMS with synthetic detections
np.random.seed(123)
boxes = np.array([
    [10, 10, 50, 50],   # box 0
    [12, 12, 52, 52],   # box 1 (overlaps with 0)
    [100, 100, 150, 150],  # box 2 (separate)
    [11, 11, 51, 51],   # box 3 (overlaps with 0)
    [200, 200, 250, 250],  # box 4 (separate)
    [98, 98, 148, 148],   # box 5 (overlaps with 2)
], dtype=np.float32)

scores = np.array([0.9, 0.75, 0.85, 0.6, 0.95, 0.7], dtype=np.float32)

kept = non_maximum_suppression(boxes, scores, iou_threshold=0.5, score_threshold=0.3)
print(f"Input boxes: {len(boxes)}")
print(f"Kept after NMS: {len(kept)} (indices: {kept})")
print(f"Kept scores: {scores[kept].tolist()}")

# Verify: no pair of kept boxes should have IoU > threshold
for i, idx_a in enumerate(kept):
    for idx_b in kept[i+1:]:
        iou = compute_iou(boxes[idx_a], boxes[idx_b])
        assert iou <= 0.5, f"NMS failed: IoU({idx_a},{idx_b}) = {iou}"

# Box 4 should be kept (highest score, no overlap)
assert 4 in kept
assert 0 in kept  # highest score in its cluster
print("\nNMS validated — no kept pairs exceed IoU threshold.")

<a id='ex6'></a>
## Exercise 6 — Batch Image Inference

Batch processing improves GPU utilization and amortizes overhead:

$$\text{Throughput} = \frac{\text{batch\_size}}{t_{\text{batch}}} \geq \frac{1}{t_{\text{single}}}$$

The improvement comes from SIMD parallelism in matrix operations.

In [ ]:
def batch_classify_images(
    images: List[np.ndarray],
    session: ort.InferenceSession,
    labels: List[str],
    target_size: Tuple[int, int] = (32, 32),
    batch_size: int = 4,
) -> List[Dict[str, Any]]:
    """Classify multiple images with batching."""
    all_results = []
    input_name = session.get_inputs()[0].name

    for i in range(0, len(images), batch_size):
        batch_imgs = images[i:i + batch_size]

        # Preprocess batch
        tensors = [preprocess_image(img, target_size) for img in batch_imgs]
        batch_tensor = np.concatenate(tensors, axis=0)  # [B, C, H, W]

        # Inference
        logits = session.run(None, {input_name: batch_tensor})[0]
        probs = softmax(logits)

        # Postprocess each image
        for j in range(len(batch_imgs)):
            pred_idx = int(np.argmax(probs[j]))
            all_results.append({
                "class_idx": pred_idx,
                "label": labels[pred_idx],
                "confidence": float(probs[j, pred_idx]),
            })

    return all_results


# Generate synthetic image dataset
n_images = 16
image_sizes = [(32, 32), (48, 64), (96, 96), (128, 128)]
images = [
    np.random.randint(0, 256, (*image_sizes[i % len(image_sizes)], 3), dtype=np.uint8)
    for i in range(n_images)
]

# Single inference timing
start = time.perf_counter()
single_results = [classify_image(img, cv_session, CIFAR_LABELS, (INPUT_SIZE, INPUT_SIZE))
                  for img in images]
single_time = (time.perf_counter() - start) * 1000

# Batch inference timing
start = time.perf_counter()
batch_results = batch_classify_images(images, cv_session, CIFAR_LABELS, (INPUT_SIZE, INPUT_SIZE), batch_size=4)
batch_time = (time.perf_counter() - start) * 1000

print(f"Single inference: {single_time:.1f} ms for {n_images} images")
print(f"Batch inference:  {batch_time:.1f} ms for {n_images} images")
print(f"Speedup: {single_time/batch_time:.2f}x")

# Verify predictions match
for s, b in zip(single_results, batch_results):
    assert s["top_prediction"] == b["label"]
print(f"\nAll {n_images} predictions match between single and batch modes.")

<a id='ex7'></a>
## Exercise 7 — Feature Extraction (Intermediate Layers)

Feature extraction accesses intermediate activations for transfer learning,
similarity search, or visualization. We modify the model to expose intermediate outputs.

In [ ]:
def build_feature_extractor(model_path: str) -> Tuple[ort.InferenceSession, List[str]]:
    """
    Modify model to expose intermediate layer outputs as additional graph outputs.
    Returns session and list of feature layer names.
    """
    model = onnx.load(model_path)

    # Find intermediate tensor names (outputs of Relu nodes)
    feature_names = []
    for node in model.graph.node:
        if node.op_type == "Relu":
            feature_names.append(node.output[0])

    # Add intermediate tensors as graph outputs
    for name in feature_names:
        feat_output = helper.make_tensor_value_info(name, TensorProto.FLOAT, None)
        model.graph.output.append(feat_output)

    # Save modified model
    feat_path = model_path.replace(".onnx", "_features.onnx")
    onnx.save(model, feat_path)

    session = ort.InferenceSession(feat_path, providers=["CPUExecutionProvider"])
    return session, feature_names


def extract_features(
    image_hwc_uint8: np.ndarray,
    session: ort.InferenceSession,
    feature_names: List[str],
    target_size: Tuple[int, int] = (32, 32),
) -> Dict[str, np.ndarray]:
    """Extract feature maps from intermediate layers."""
    tensor = preprocess_image(image_hwc_uint8, target_size=target_size)
    input_name = session.get_inputs()[0].name

    all_names = [o.name for o in session.get_outputs()]
    outputs = session.run(all_names, {input_name: tensor})

    features = {}
    for name, out in zip(all_names, outputs):
        if name in feature_names:
            features[name] = np.asarray(out)

    return features


# Build feature extractor
feat_session, feat_names = build_feature_extractor(MODEL_PATH)
print(f"Feature layers exposed: {feat_names}")
print(f"Total outputs: {len(feat_session.get_outputs())}")

# Extract features
features = extract_features(synthetic_image, feat_session, feat_names, (INPUT_SIZE, INPUT_SIZE))

print("\nExtracted feature shapes:")
for name, feat in features.items():
    print(f"  {name}: {feat.shape}")
    # Compute feature statistics
    print(f"    mean={feat.mean():.4f}, std={feat.std():.4f}, sparsity={100*(feat==0).mean():.1f}%")

# Verify features have expected properties
assert len(features) == len(feat_names)
for feat in features.values():
    assert feat.ndim == 4  # [batch, channels, H, W]
    assert feat.min() >= 0  # After ReLU
print("\nFeature extraction validated (all activations non-negative after ReLU).")

<a id='challenge'></a>
## Challenge — Complete Image Classification Pipeline

Build a production-ready pipeline that combines all components:
preprocessing, inference, postprocessing, feature caching, and metrics.

In [ ]:
class CVInferencePipeline:
    """Production computer vision inference pipeline."""

    def __init__(
        self,
        model_path: str,
        labels: List[str],
        input_size: Tuple[int, int] = (32, 32),
    ):
        self.session = build_cv_session(model_path)
        self.labels = labels
        self.input_size = input_size
        self.input_name = self.session.get_inputs()[0].name
        self.metrics = {
            "total_images": 0,
            "total_latency_ms": 0.0,
            "class_distribution": {l: 0 for l in labels},
        }

    def predict_single(self, image: np.ndarray, top_k: int = 5) -> Dict[str, Any]:
        start = time.perf_counter()
        tensor = preprocess_image(image, self.input_size)
        logits = self.session.run(None, {self.input_name: tensor})[0][0]
        probs = softmax(logits)
        latency = (time.perf_counter() - start) * 1000

        top_indices = np.argsort(-probs)[:top_k]
        pred_idx = top_indices[0]
        label = self.labels[pred_idx]

        self.metrics["total_images"] += 1
        self.metrics["total_latency_ms"] += latency
        self.metrics["class_distribution"][label] += 1

        return {
            "label": label,
            "confidence": float(probs[pred_idx]),
            "top_k": [
                {"label": self.labels[i], "prob": float(probs[i])}
                for i in top_indices
            ],
            "latency_ms": round(latency, 3),
        }

    def predict_batch(self, images: List[np.ndarray]) -> List[Dict[str, Any]]:
        start = time.perf_counter()
        tensors = np.concatenate(
            [preprocess_image(img, self.input_size) for img in images], axis=0
        )
        logits = self.session.run(None, {self.input_name: tensors})[0]
        probs = softmax(logits)
        latency = (time.perf_counter() - start) * 1000

        results = []
        for j in range(len(images)):
            pred_idx = int(np.argmax(probs[j]))
            label = self.labels[pred_idx]
            self.metrics["class_distribution"][label] += 1
            results.append({
                "label": label,
                "confidence": float(probs[j, pred_idx]),
            })

        self.metrics["total_images"] += len(images)
        self.metrics["total_latency_ms"] += latency
        return results

    def get_metrics(self) -> Dict[str, Any]:
        avg_lat = self.metrics["total_latency_ms"] / max(self.metrics["total_images"], 1)
        return {
            **self.metrics,
            "avg_latency_ms": round(avg_lat, 3),
            "throughput_ips": round(1000 / avg_lat, 1) if avg_lat > 0 else 0,
        }


# Test the pipeline
pipeline = CVInferencePipeline(MODEL_PATH, CIFAR_LABELS, input_size=(INPUT_SIZE, INPUT_SIZE))

# Single predictions
for img in images[:5]:
    r = pipeline.predict_single(img)
    print(f"  {r['label']:12s} conf={r['confidence']:.3f} latency={r['latency_ms']:.2f}ms")

# Batch prediction
batch_results = pipeline.predict_batch(images[5:])
print(f"\nBatch processed {len(batch_results)} images")

# Metrics
metrics = pipeline.get_metrics()
print(f"\nPipeline Metrics:")
print(f"  Total images: {metrics['total_images']}")
print(f"  Avg latency: {metrics['avg_latency_ms']:.2f} ms")
print(f"  Throughput: {metrics['throughput_ips']:.0f} images/sec")

assert metrics["total_images"] == n_images
print("\nCV inference pipeline validated!")

<a id='summary'></a>
## Summary

| Component | Key Technique |
|-----------|---------------|
| **Preprocessing** | Resize → float32 → normalize (ImageNet stats) → NCHW |
| **ResNet Block** | Skip connection: $y = F(x) + x$ |
| **Classification** | Softmax with temperature → top-k |
| **NMS** | IoU-based suppression for detection outputs |
| **Batching** | Concatenate along batch dim for throughput |
| **Feature Extraction** | Expose intermediate outputs for transfer learning |

**Key equations:**

$$\text{IoU}(A,B) = \frac{|A \cap B|}{|A \cup B|} \qquad \text{Residual: } y = \mathcal{F}(x) + x$$